In [1]:
from dotenv import load_dotenv
import anthropic, json, pandas as pd
from pathlib import Path
import os

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [3]:
load_dotenv()
client = anthropic.Anthropic()  # lee ANTHROPIC_API_KEY del entorno

In [4]:
df = pd.read_csv(root / "csv_enunciados/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [37]:
df[df.asignatura.str.startswith("Mates")].tema.unique()

<ArrowStringArray>
[        'Contraste De Hipótesis',          'Distribución Binomial',
                  'Funciones CCSS',         'Inferencia Estadística',
        'Matrices Y Determinantes',                    'Probabilidad',
            'Programación Lineal', 'Sistemas de Ecuaciones Lineales',
 'Distribución normal y binomial',                       'Funciones',
                      'Geometría',                      'Integrales',
 'Sistemas De Ecuaciones Lineales']
Length: 13, dtype: str

In [5]:
sample = df.loc[df.tema == "Campo gravitatorio", ["id", "enunciado"]]
sample

,id,enunciado
121,fisica_campo_gravitatorio_2026_junio_ejercicio_a,Razone la veracidad de las siguientes afirmaci...
122,fisica_campo_gravitatorio_2026_junio_ejercicio_b,Un niño de 15 kg resbala desde el reposo a lo ...
123,fisica_campo_gravitatorio_2025_julio_ejercicio_a,Discuta razonadamente si los siguientes enunci...
124,fisica_campo_gravitatorio_2025_julio_ejercicio_b1,Dos masas puntuales de 200 kg están situadas e...
125,fisica_campo_gravitatorio_2025_julio_ejercicio_b2,Un satélite de 1400 kg en una órbita circular ...
...,...,...
239,fisica_campo_gravitatorio_2016_reserva_3_ejerc...,El satélite español PAZ de observación de la T...
240,fisica_campo_gravitatorio_2016_reserva_4_ejerc...,a) Enuncie la ley de gravitación universal y c...
241,fisica_campo_gravitatorio_2016_reserva_4_ejerc...,Un bloque de 5 kg desliza por una superficie h...
242,fisica_campo_gravitatorio_2016_septiembre_ejer...,a) Energía potencial asociada a una fuerza con...


In [9]:
sample.loc[123, "enunciado"]

'Discuta razonadamente si los siguientes enunciados son verdaderos o falsos: i) si se realiza\ntrabajo sobre una partícula, su energía cinética aumenta; ii) las fuerzas conservativas siempre\nrealizan trabajo nulo.'

In [6]:
def classify(enunciado, system_md, modelo="claude-haiku-4-5-20251001"):
    resp = client.messages.create(
        model=modelo,
        max_tokens=300,
        system=[{"type": "text", "text": system_md,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": f"Enunciado:\n{enunciado}"}],
    )
    texto = resp.content[0].text.strip().removeprefix("```json").removesuffix("```").strip()
    return json.loads(texto)

In [10]:
system_md = (root / "classifier/campo_gravitatorio.md").read_text(encoding="utf-8")

In [8]:
output_file = root / "classifier/resultados.jsonl"
already_processed = {json.loads(l)["id"] for l in output_file.open()} if output_file.exists() else set()

In [ ]:
# for index, row in sample.iterrows():
#     if row.id in already_processed:
#         continue
#     try:
#         r = classify(row.enunciado, system_md)
#         r["id"] = row.id
#     except Exception as e:
#         r = {"id": row.id, "error": str(e)}
#     # important!! use write mode = "a" to add
#     # classification of each row to results file
#     with output_file.open("a", encoding="utf-8") as f:
#         f.write(json.dumps(r, ensure_ascii=False) + "\n")